In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import MobileNetV2

# Adjust local logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- 1. Prepare and Preprocess the Dataset (Using Built-in CIFAR-10) ---
print("Loading built-in dataset safely from Keras local repository...")
# CIFAR-10 has 10 classes and contains 32x32 images
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

num_classes = 10
print(f"Number of classes in dataset: {num_classes}")

# Simple preprocessing function to resize and normalize images
def preprocess_image_pipeline(images, labels):
    # Convert numpy arrays to TensorFlow tensors
    images = tf.cast(images, dtype=tf.float32)
    # Normalize pixel values from [0, 255] to [0, 1]
    images = images / 255.0
    # Upscale images from 32x32 to your exact requested MobileNet shape (96x96)
    images = tf.image.resize(images, (96, 96))
    return images, labels

# Build highly optimized tf.data input streams
batch_size = 32

ds_train = tf.data.Dataset.from_tensor_slices((x_train_raw, y_train_raw))
ds_train = ds_train.shuffle(5000).batch(batch_size)
ds_train = ds_train.map(preprocess_image_pipeline, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

ds_test = tf.data.Dataset.from_tensor_slices((x_test_raw, y_test_raw))
ds_test = ds_test.batch(batch_size)
ds_test = ds_test.map(preprocess_image_pipeline, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


# --- 2. Load Pre-trained CNN Model (Step a) ---
print("\nLoading pre-trained MobileNetV2 base network with custom shape...")
# Integration of your requested base_model parameters
base_model = MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    input_shape=(96, 96, 3)
)


# --- 3. Freeze Parameters in Lower Layers (Step b) ---
print("Freezing lower convolutional weights...")
base_model.trainable = False 


# --- 4. Add Custom Trainable Classifier (Step c) ---
print("Building custom structural network architecture...")
inputs = tf.keras.Input(shape=(96, 96, 3), name="Input_Image")

# Pass inputs through the frozen base feature extractor
x = base_model(inputs, training=False)

# Convert feature maps to a standard 1D feature vector
x = tf.keras.layers.GlobalAveragePooling2D(name="Global_Average_Pooling")(x)

# Custom classifier layers with trainable parameters
x = tf.keras.layers.Dense(256, activation='relu', name="Dense_Trainable_1")(x)
x = tf.keras.layers.Dropout(0.3, name="Dropout_Regularization")(x)

# Output classification layer corresponding to new dataset classes
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name="Output_Predictions")(x)

# Assemble structural model boundaries
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Transfer_Learning_Classifier")

# Compile the setup
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


# --- 5. Train Classifier Layers (Step d) ---
# Keeping epochs short (2) so it executes quickly during your lab evaluation
epochs_initial = 2
print(f"\nTraining custom classifier layers for {epochs_initial} epochs...")
history = model.fit(
    ds_train,
    epochs=epochs_initial,
    validation_data=ds_test
)


# --- 6. Fine-Tuning: Unfreeze More Layers & Train (Step e) ---
print("\n[Fine-Tuning Process Started]")
# Unfreeze base model weights
base_model.trainable = True

# Refened constraint: Keep lower layers frozen, unfreeze top convolutional layers
# MobileNetV2 contains 154 layers. We unfreeze everything from layer 100 onwards.
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile the model with a lower learning rate to prevent exploding gradients
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Updated layer trainability statuses:")
print(f"Base Model Layers: {len(base_model.layers)} | Trainable Layers: {len(base_model.layers) - fine_tune_at}")

epochs_fine = 2
total_epochs = epochs_initial + epochs_fine

print(f"\nFine-tuning model for an additional {epochs_fine} epochs...")
history_fine = model.fit(
    ds_train,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    validation_data=ds_test
)
print("\nTraining workflow execution complete.")


Loading built-in dataset safely from Keras local repository...
A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
  3186688/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:27:18 31us/step